# nbS1 slice neuron sets based on a hexgrid parcellation

Copyright (c) 2026 Open Brain Institute

Authors: Christoph Pokorny

Last modified: 04.2026

## Summary
This notebook allows a user to interactively select and visualize slice neuron sets based on a pre-computed hexagonal parcellation of the `nbS1` circuit. The pre-computed parcellation from the [ConnectomeUtilities](https://github.com/openbraininstitute/ConnectomeUtilities/blob/main/examples/data/voxel-based-hex-grid-info-with-conicality.h5) repo is used. For details, see the [README](README.md).

In [ ]:
import bluepysnap as snap
import json
import numpy as np
import pandas
import plotly.express as px
import plotly.graph_objects as go
import random

from conntility.circuit_models import neuron_groups
from entitysdk import Client, LocalAssetStore, models
from entitysdk.staging import stage_circuit
from ipywidgets import widgets
from obi_auth import get_token
from obi_notebook import get_projects
from obi_notebook.get_environment import get_environment
from pathlib import Path

In [ ]:
column_prop_fn = "./voxel-based-hex-grid-info-with-conicality.h5"
column_ids_fn = "./column_identities.nrrd"

## Project and circuit selection

As a first step we select one of the projects we have access to that the circuit is associated with. Since the circuit of interest is part of the public OBI circuits, any project can be selected.

### Project selection

In [ ]:
token = get_token(environment=get_environment(), auth_mode="daf")
project_context = get_projects.get_projects(token)

In [ ]:
client = Client(environment=get_environment(), project_context=project_context, token_manager=token, local_store=LocalAssetStore(prefix="/data"))

### nbS1 circuit selection

In [ ]:
# Fetch circuit entity
circuit_name = "nbS1"

fetched = client.search_entity(entity_type=models.Circuit, query={"name": circuit_name}).one()
print(f"Circuit fetched: {fetched.name} (ID {fetched.id})\n")
print(f"#Neurons: {fetched.number_neurons}, #Synapses: {fetched.number_synapses}, #Connections: {fetched.number_connections}\n")
print(f"{fetched.description}\n")

In [ ]:
# Make circuit available
output_dir = Path(f"./{circuit_name}")
if output_dir.exists():
    circ_fn = output_dir / "circuit_config.json"
    if not circ_fn.is_file():
        msg = f"Circuit config not found! Delete '{output_dir.name}' folder and fetch circuit again."
        raise FileNotFoundError(msg)
else:
    circ_fn = stage_circuit(
        client=client,
        model=fetched,
        output_dir=output_dir,
    )
print(f"Circuit '{circuit_name}' staged at '{output_dir.resolve()}'.")

In [ ]:
# Load circuit & base grouping
circ = snap.Circuit(circ_fn)
popul = "S1nonbarrel_neurons"
loader_config = {
    "loading": {
        "base_target": "All", 
        "node_population": popul,
        "properties": ["x", "y", "z", "etype", "mtype", "layer", "synapse_class"],
        "atlas": [
            {"data": column_ids_fn,
             "properties": ["column_id"]}
        ]
    },
    # Optional filtering
    # "filtering":[
    #     {
    #         "column": "layer",
    #         "value": 4,
    #     }
    # ],
    "grouping": [
        {
            "method": "group_by_properties",
            "columns": ["column_id"]
        }
    ]
}

base_grp = neuron_groups.load_group_filter(circ, loader_config)

print(f"Loaded circuit with {base_grp.shape[0]} neurons")
display(base_grp)
# NOTE: column_id 0 indicates a node is NOT MEMBER OF ANY COLUMN!

### Hexgrid column selection & visualization

We first define the longest slice consisting of hex columns along the diagonal of the `nbS1` circuit. Then, the user can select the length of the slice (number of hex columns).

In [ ]:
# Load hexgrid column properties
col_props = pandas.read_hdf(column_prop_fn, "grid-info").set_index("nrrd-file-id")
display(col_props)

In [ ]:
# Longest slice definition
R = [str(r) for r in range(18, 0, -1)]  # Rows 18, 17, ..., 1
C = np.repeat([str(c) for c in range(1, 10, 1)], 2)  # Columns 1, 1, 2, 2, ..., 9, 9
subtarget_slice = [f"R{R};C{C}" for R, C in zip(R, C)]
col_props_slice = col_props.set_index("grid-subtarget")
col_props_slice["column_id"] = col_props.index
col_props_slice = col_props_slice.loc[subtarget_slice]
print(f"{col_props_slice.shape[0]} columns in longest slice: {col_props_slice["column_id"].index.to_numpy()}")


In [ ]:
# Select slice with selected number of hex subtargets
num_hex_list = list(range(1, len(subtarget_slice) + 1))
hex_wdgt = widgets.Dropdown(options=num_hex_list, description="Select number of hex sub-targets in slice:", value=num_hex_list[-1], style={"description_width": "auto"}, layout=widgets.Layout(width="max-content"))
display(hex_wdgt)

In [ ]:
# Plot selected slice in 2D
sel_idx = int(np.round(len(num_hex_list) / 2 - hex_wdgt.value / 2))  # Select from center
subtarget_slice_sel = subtarget_slice[sel_idx : sel_idx + hex_wdgt.value]
col_props_slice_sel = col_props_slice.loc[subtarget_slice_sel]
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=col_props["grid-x"],
    y=col_props["grid-y"],
    mode="markers",
    marker=dict(symbol="hexagon", size=35, color="cornflowerblue", opacity=0.2),
))

fig.add_trace(go.Scatter(
    x=col_props_slice_sel["grid-x"],
    y=col_props_slice_sel["grid-y"],
    mode="markers+text",
    marker=dict(symbol="hexagon", size=35, color="cornflowerblue", opacity=1.0),
    text=col_props_slice_sel.index,
    textposition="middle center",
    textfont=dict(size=7, color="black"),
))

fig.update_layout(
    width=800,
    height=800,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False, scaleanchor="x"),
    title=dict(text="2D hex grid with slice sub-targets", x=0.5, xanchor="center", y=0.87),
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.show()

In [ ]:
# Plot selected slice in 3D
base_grp_plot = base_grp.sample(frac=0.05, random_state=0)  # Subsample for plotting; increase for higher resolution
base_grp_plot["column_id_category"] = base_grp_plot["column_id"].astype(str)  # To be treated as categorical column

colors = list(px.colors.qualitative.Alphabet)
random.seed(0)  # Change seed for different randomization of colors
random.shuffle(colors)

fig = go.Figure()

fig = px.scatter_3d(base_grp_plot, x="x", y="y", z="z", color="column_id_category", opacity=0.1)
fig.update_traces(marker=dict(color="grey"))
base_grp_plot_sel = base_grp_plot[np.isin(base_grp_plot["column_id"], col_props_slice_sel["column_id"])]
fig_sel = px.scatter_3d(base_grp_plot_sel, x="x", y="y", z="z", color="column_id_category", opacity=1.0, color_discrete_sequence=colors)
for trace in fig_sel.data:
    fig.add_trace(trace)

fig.update_traces(marker=dict(size=1))
fig.update_layout(
    width=800,
    height=800,
    title=dict(text="3D hex grid with slice sub-targets", x=0.5, xanchor="center", y=0.87),
    showlegend=False
    )

fig.show()

### Getting the actual neuron IDs

The list of neuron IDs are written to a node_sets.json file in a SONATA compatible format. Optionally, they can be copy-pasted directly from this notebook.

In [ ]:
# Get all selected neurons
base_grp_sel = base_grp.loc[np.isin(base_grp["column_id"], col_props_slice_sel["column_id"])]
print(f"Total selected count: {base_grp_sel.shape[0]} neurons")
print(f"Average count per column: {base_grp_sel.shape[0]/len(subtarget_slice_sel):.1f} neurons")

# Write to .json file
node_set_dict = {
    "custom_node_set": {
        "population": popul,
        "node_id": base_grp_sel["node_ids"].to_list(),
        }
    }
node_sets_file = Path("node_sets.json")
with node_sets_file.open("w") as f:
    json.dump(node_set_dict, f, indent=None)
print(f"\nNeuron IDs written to '{node_sets_file}'")

In [ ]:
# OPTIONAL: Print neuron IDs for copy-pasting them
# out_wdgt = widgets.Output(layout={"max_height": "100px", "overflow": "scroll"})
# with out_wdgt:
#     print(base_grp_sel["node_ids"].to_list())
# display(out_wdgt)

### Plot neuron counts per hex column

In [ ]:
# Neuron counts per column
neuron_counts = base_grp_sel[["node_ids", "column_id"]].groupby("column_id").count()
neuron_counts.columns = ["neuron-count"]
neuron_counts["grid-subtarget"] = col_props["grid-subtarget"].loc[neuron_counts.index]
neuron_counts = neuron_counts.loc[col_props_slice_sel["column_id"]]
neuron_counts

In [ ]:
# Plot neuron counts per column
fig = px.bar(neuron_counts.reset_index(), x="grid-subtarget", y="neuron-count")
fig.update_layout(
    title=dict(text="Neuron count per sub-target", x=0.5, xanchor="center"),
    xaxis_title="Sub-target",
    yaxis_title="Neuron count",
)
fig.show()